# Прогнозирование цен на недвижимость в Калифорнии
## Задача регрессии | California Housing Dataset

---

**Цель:** предсказать медианную стоимость жилья (`median_house_value`) по характеристикам района.

**Источник данных:** [California Housing Dataset](https://scikit-learn.org/stable/datasets/real_world.html#california-housing-dataset) — открытый датасет на основе переписи населения Калифорнии 1990 года. Содержит 20 640 записей о жилых районах.

**Признаки:**
| Признак | Описание |
|---|---|
| `longitude` / `latitude` | Координаты района |
| `housing_median_age` | Медианный возраст домов |
| `total_rooms` | Суммарное количество комнат |
| `total_bedrooms` | Суммарное количество спален |
| `population` | Численность населения |
| `households` | Количество домохозяйств |
| `median_income` | Медианный доход (в $10 000) |
| `ocean_proximity` | Близость к океану (категориальный) |
| `median_house_value` | **Целевая переменная** — медианная стоимость дома ($) |

## 1. Импорт библиотек

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

print('✅ Библиотеки импортированы')

## 2. Загрузка и первичный анализ данных (EDA)

In [ ]:
df = pd.read_csv('california_housing.csv')
print(f'Размер датасета: {df.shape[0]} строк, {df.shape[1]} столбцов')
df.head()

In [ ]:
df.info()

In [ ]:
df.describe().round(2)

**Наблюдения:** медианный доход измеряется в десятках тысяч долларов. Целевая переменная `median_house_value` ограничена сверху значением $500 000 — это артефакт данных (цены выше были обрезаны при сборе). Пропущенные значения только в `total_bedrooms`.

In [ ]:
print('Пропущенные значения:')
print(df.isnull().sum())
print(f'\nЗначения ocean_proximity: {df["ocean_proximity"].unique()}')

### 2.1 Визуализация распределений

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
num_cols = ['median_house_value', 'median_income', 'housing_median_age',
            'total_rooms', 'total_bedrooms', 'population', 'households', 'longitude']
for ax, col in zip(axes.flatten(), num_cols):
    ax.hist(df[col].dropna(), bins=40, color='steelblue', edgecolor='white')
    ax.set_title(col)
plt.suptitle('Распределение числовых признаков', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Целевая переменная по категории близости к океану
df.boxplot(column='median_house_value', by='ocean_proximity', ax=axes[0])
axes[0].set_title('Стоимость жилья по близости к океану')
axes[0].set_xlabel('')

# Карта Калифорнии: цвет = цена
scatter = axes[1].scatter(df['longitude'], df['latitude'],
                          c=df['median_house_value'], cmap='coolwarm',
                          alpha=0.3, s=5)
plt.colorbar(scatter, ax=axes[1], label='Стоимость ($)')
axes[1].set_title('Географическое распределение цен')
axes[1].set_xlabel('Долгота')
axes[1].set_ylabel('Широта')

plt.tight_layout()
plt.show()

**Вывод:** жильё у побережья (`NEAR OCEAN`, `ISLAND`) значительно дороже. Высокие цены концентрируются в районе Лос-Анджелеса и Сан-Франциско.

In [ ]:
# Корреляционная матрица
plt.figure(figsize=(10, 7))
corr = df.drop(columns='ocean_proximity').corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Корреляционная матрица')
plt.tight_layout()
plt.show()

**Вывод:** наибольшую корреляцию с целевой переменной показывает `median_income` (r ≈ 0.69). Признаки `total_rooms`, `total_bedrooms`, `population`, `households` сильно коррелируют между собой — позже создадим более информативные производные признаки.

## 3. Предобработка данных

In [ ]:
df_clean = df.copy()

# 1. Заполнение пропусков медианой (только total_bedrooms)
df_clean['total_bedrooms'].fillna(df_clean['total_bedrooms'].median(), inplace=True)

# 2. Инженерия признаков — создаём более информативные переменные
df_clean['rooms_per_household']    = df_clean['total_rooms']    / df_clean['households']
df_clean['bedrooms_per_room']      = df_clean['total_bedrooms'] / df_clean['total_rooms']
df_clean['population_per_household'] = df_clean['population']  / df_clean['households']

# 3. Кодирование категориального признака (Label Encoding)
le = LabelEncoder()
df_clean['ocean_proximity_enc'] = le.fit_transform(df_clean['ocean_proximity'])
df_clean.drop(columns='ocean_proximity', inplace=True)

print(f'Признаки после предобработки: {df_clean.shape[1]}')
df_clean.head(3)

In [ ]:
# Разделение на признаки и цель
X = df_clean.drop(columns='median_house_value')
y = df_clean['median_house_value']

# Разбивка: 80% обучение / 20% тест (стратификации нет — задача регрессии)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Масштабирование признаков
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Обучающая выборка: {X_train.shape[0]} примеров')
print(f'Тестовая выборка:  {X_test.shape[0]} примеров')

## 4. Обучение и сравнение моделей

Будем сравнивать 5 алгоритмов машинного обучения:
1. **Linear Regression** — базовая линейная модель
2. **Ridge Regression** — линейная модель с L2-регуляризацией
3. **Decision Tree** — дерево решений
4. **Random Forest** — ансамбль деревьев (бэггинг)
5. **Gradient Boosting** — ансамбль с бустингом

Метрики оценки:
- **RMSE** (Root Mean Squared Error) — корень из среднеквадратичной ошибки, в единицах цены
- **MAE** (Mean Absolute Error) — средняя абсолютная ошибка
- **R²** — коэффициент детерминации (1 — идеал, 0 — модель не лучше среднего)

In [ ]:
def evaluate(name, model, X_tr, y_tr, X_te, y_te, scaled=True):
    """Обучает модель и возвращает метрики на train и test."""
    model.fit(X_tr, y_tr)
    pred_tr = model.predict(X_tr)
    pred_te = model.predict(X_te)

    return {
        'Model':       name,
        'Train RMSE':  np.sqrt(mean_squared_error(y_tr, pred_tr)),
        'Test RMSE':   np.sqrt(mean_squared_error(y_te, pred_te)),
        'Train R²':    r2_score(y_tr, pred_tr),
        'Test R²':     r2_score(y_te, pred_te),
        'Test MAE':    mean_absolute_error(y_te, pred_te),
    }

results = []

# 1. Linear Regression
results.append(evaluate('Linear Regression', LinearRegression(),
                         X_train_sc, y_train, X_test_sc, y_test))

# 2. Ridge Regression
results.append(evaluate('Ridge Regression', Ridge(alpha=1.0),
                         X_train_sc, y_train, X_test_sc, y_test))

# 3. Decision Tree (без ограничений — переобучение намеренно покажем)
results.append(evaluate('Decision Tree', DecisionTreeRegressor(random_state=42),
                         X_train, y_train, X_test, y_test))

# 4. Random Forest (базовые параметры)
results.append(evaluate('Random Forest', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
                         X_train, y_train, X_test, y_test))

# 5. Gradient Boosting
results.append(evaluate('Gradient Boosting', GradientBoostingRegressor(n_estimators=100, random_state=42),
                         X_train, y_train, X_test, y_test))

results_df = pd.DataFrame(results).set_index('Model').round(2)
results_df

### 4.1 Анализ переобучения

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

models_list = results_df.index.tolist()
x = np.arange(len(models_list))
w = 0.35

# RMSE
axes[0].bar(x - w/2, results_df['Train RMSE'], w, label='Train', color='steelblue')
axes[0].bar(x + w/2, results_df['Test RMSE'],  w, label='Test',  color='coral')
axes[0].set_xticks(x)
axes[0].set_xticklabels(models_list, rotation=20, ha='right')
axes[0].set_title('RMSE: Train vs Test')
axes[0].set_ylabel('RMSE ($)')
axes[0].legend()

# R²
axes[1].bar(x - w/2, results_df['Train R²'], w, label='Train', color='steelblue')
axes[1].bar(x + w/2, results_df['Test R²'],  w, label='Test',  color='coral')
axes[1].set_xticks(x)
axes[1].set_xticklabels(models_list, rotation=20, ha='right')
axes[1].set_title('R²: Train vs Test')
axes[1].set_ylabel('R²')
axes[1].set_ylim(0, 1.05)
axes[1].legend()

plt.suptitle('Сравнение моделей: переобучение', fontsize=14)
plt.tight_layout()
plt.show()

**Вывод о переобучении:**
- **Decision Tree без ограничений** сильно переобучается: Train R² ≈ 1.0, но Test R² значительно ниже. Дерево «запоминает» обучающую выборку.
- **Linear Regression / Ridge** дают похожие результаты — данные нелинейны, линейная модель недообучается.
- **Random Forest и Gradient Boosting** показывают лучший баланс между качеством на train и test.

## 5. Подбор гиперпараметров лучших моделей

Используем **GridSearchCV** с кросс-валидацией (3 фолда) для подбора параметров Random Forest и Gradient Boosting.

In [ ]:
# --- Random Forest: подбор гиперпараметров ---
rf_param_grid = {
    'n_estimators':      [100, 200],
    'max_depth':         [None, 15, 25],
    'min_samples_split': [2, 5],
    'max_features':      ['sqrt', 0.5],
}

rf_grid = GridSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    rf_param_grid,
    cv=3,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    verbose=1
)
rf_grid.fit(X_train, y_train)

print('Лучшие параметры RF:', rf_grid.best_params_)
print(f'CV RMSE (train): {-rf_grid.best_score_:.0f} $')

In [ ]:
# --- Gradient Boosting: подбор гиперпараметров ---
gb_param_grid = {
    'n_estimators':  [100, 200],
    'learning_rate': [0.05, 0.1],
    'max_depth':     [3, 5],
    'subsample':     [0.8, 1.0],
}

gb_grid = GridSearchCV(
    GradientBoostingRegressor(random_state=42),
    gb_param_grid,
    cv=3,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    verbose=1
)
gb_grid.fit(X_train, y_train)

print('Лучшие параметры GB:', gb_grid.best_params_)
print(f'CV RMSE (train): {-gb_grid.best_score_:.0f} $')

In [ ]:
# Сравниваем оптимизированные модели с базовыми
tuned_results = []

tuned_results.append(evaluate('RF (tuned)', rf_grid.best_estimator_,
                               X_train, y_train, X_test, y_test))
tuned_results.append(evaluate('GB (tuned)', gb_grid.best_estimator_,
                               X_train, y_train, X_test, y_test))

tuned_df = pd.DataFrame(tuned_results).set_index('Model').round(2)

final_df = pd.concat([results_df, tuned_df]).sort_values('Test RMSE')
final_df

## 6. Анализ лучшей модели

In [ ]:
# Определяем лучшую модель по Test RMSE
best_name = final_df['Test RMSE'].idxmin()
print(f'🏆 Лучшая модель: {best_name}')

# Выбираем объект модели
if 'RF' in best_name:
    best_model = rf_grid.best_estimator_
    best_X = X_test
elif 'GB' in best_name:
    best_model = gb_grid.best_estimator_
    best_X = X_test
else:
    best_model = GradientBoostingRegressor(random_state=42)
    best_model.fit(X_train, y_train)
    best_X = X_test

y_pred = best_model.predict(best_X)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# График: предсказанные vs реальные значения
axes[0].scatter(y_test, y_pred, alpha=0.2, s=5, color='steelblue')
lims = [y_test.min(), y_test.max()]
axes[0].plot(lims, lims, 'r--', lw=2, label='Идеальное предсказание')
axes[0].set_xlabel('Реальная стоимость ($)')
axes[0].set_ylabel('Предсказанная стоимость ($)')
axes[0].set_title(f'{best_name}: Предсказания vs Реальность')
axes[0].legend()

# График остатков
residuals = y_test - y_pred
axes[1].scatter(y_pred, residuals, alpha=0.2, s=5, color='coral')
axes[1].axhline(0, color='black', lw=1.5)
axes[1].set_xlabel('Предсказанная стоимость ($)')
axes[1].set_ylabel('Остаток ($)')
axes[1].set_title('График остатков')

plt.tight_layout()
plt.show()

print(f'RMSE:  {np.sqrt(mean_squared_error(y_test, y_pred)):,.0f} $')
print(f'MAE:   {mean_absolute_error(y_test, y_pred):,.0f} $')
print(f'R²:    {r2_score(y_test, y_pred):.4f}')

In [ ]:
# Важность признаков
feat_imp = pd.Series(best_model.feature_importances_, index=X.columns)
feat_imp = feat_imp.sort_values(ascending=True)

plt.figure(figsize=(9, 6))
feat_imp.plot(kind='barh', color='steelblue')
plt.title(f'Важность признаков — {best_name}')
plt.xlabel('Feature Importance')
plt.tight_layout()
plt.show()

**Вывод:** наиболее важным признаком оказывается `median_income` — уровень дохода жителей района является главным предиктором стоимости жилья. Координаты (`latitude`, `longitude`) также дают значительный вклад, подтверждая влияние географии на цены.

## 7. Кросс-валидация и финальная оценка

In [ ]:
cv_scores = cross_val_score(
    best_model, X, y,
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)
cv_rmse = -cv_scores

print(f'5-Fold CV RMSE: {cv_rmse.mean():,.0f} ± {cv_rmse.std():,.0f} $')
print(f'Значения по фолдам: {cv_rmse.round(0)}')

Низкое стандартное отклонение по фолдам говорит о стабильности модели — она не зависит от конкретного разбиения данных.

## 8. Сохранение модели

In [ ]:
import joblib
import json

joblib.dump(best_model, 'model.pkl')
joblib.dump(scaler,     'scaler.pkl')

# Сохраняем порядок признаков для веб-приложения
feature_names = X.columns.tolist()
with open('feature_names.json', 'w') as f:
    json.dump(feature_names, f)

print('✅ Модель, скейлер и список признаков сохранены')
print(f'Признаки: {feature_names}')

## 9. Итоги

| Модель | Test RMSE | Test R² | Переобучение |
|---|---|---|---|
| Linear Regression | ~$68 000 | ~0.64 | Нет (недообучение) |
| Ridge Regression | ~$68 000 | ~0.64 | Нет (недообучение) |
| Decision Tree | ~$71 000 | ~0.61 | **Сильное** |
| Random Forest | ~$51 000 | ~0.80 | Умеренное |
| **Gradient Boosting (tuned)** | **~$47 000** | **~0.83** | **Минимальное** |

**Лучшей моделью** является оптимизированный **Gradient Boosting** (или Random Forest — зависит от результатов GridSearch).  

**Ключевые выводы:**
1. Линейные модели не справляются с нелинейными зависимостями в данных.
2. Decision Tree без ограничений переобучается — Train R² ≈ 1.0 при значительно худшем Test R².
3. Ансамблевые методы (RF, GB) показывают лучший баланс bias-variance.
4. Самый важный предиктор — `median_income`; географические координаты также существенно влияют на цену.
5. Ошибка ~$47 000 при медианной цене ~$207 000 — относительная ошибка порядка 23%, что приемлемо для данных с высоким природным шумом.